# 01 - EDA & Hiểu dữ liệu: Pima Indians Diabetes Database

**Mục tiêu notebook này:**
- Load và xem tổng quan dataset
- Kiểm tra missing values (ẩn dưới dạng 0)
- Phân tích phân bố từng biến
- Phân tích mối quan hệ với Outcome (target)
- Phát hiện outlier
- Correlation & heatmap
- Kết luận insights để preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cài đặt style đẹp hơn
plt.style.use('seaborn')
sns.set_palette('viridis')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load dữ liệu

In [ ]:
DATA_PATH = "../data/raw/diabetes.csv"  # thay đổi nếu đường dẫn khác

df = pd.read_csv(DATA_PATH)
print("Kích thước dataset:", df.shape)
df.head()

In [ ]:
# Thông tin cơ bản
df.info()

## 2. Thống kê mô tả

In [ ]:
df.describe().T

**Quan sát ban đầu:**
- Glucose, BloodPressure, SkinThickness, Insulin, BMI có giá trị min = 0 → rất có thể là **missing values ẩn** (không hợp lý về mặt y khoa).
- Outcome: 0/1 → bài toán phân loại nhị phân (imbalance nhẹ).

## 3. Kiểm tra missing values (ẩn dưới dạng 0)

In [ ]:
cols_with_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("Số lượng giá trị 0 ở các cột nghi ngờ missing:")
for col in cols_with_zero:
    zeros = (df[col] == 0).sum()
    print(f"{col}: {zeros} giá trị 0 ({zeros/len(df)*100:.2f}%)")

## 4. Phân bố từng biến (Histogram + Boxplot)

In [ ]:
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

plt.figure(figsize=(16, 12))
for i, col in enumerate(features, 1):
    plt.subplot(4, 2, i)
    sns.histplot(df[col], kde=True, color='teal')
    plt.title(f'Phân bố của {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot theo Outcome
plt.figure(figsize=(16, 12))
for i, col in enumerate(features, 1):
    plt.subplot(4, 2, i)
    sns.boxplot(x='Outcome', y=col, data=df, palette='viridis')
    plt.title(f'{col} theo Outcome')
plt.tight_layout()
plt.show()

## 5. Tỷ lệ Outcome (class imbalance)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Outcome', data=df, palette='viridis')
plt.title('Phân bố Outcome (0: Không tiểu đường, 1: Có tiểu đường)')
plt.show()

print(df['Outcome'].value_counts(normalize=True) * 100)

## 6. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Ma trận tương quan giữa các biến')
plt.show()

## 7. Pairplot (quan hệ đôi giữa các biến quan trọng)

In [ ]:
important_cols = ['Glucose', 'BMI', 'Age', 'Insulin', 'Outcome']
sns.pairplot(df[important_cols], hue='Outcome', palette='viridis', diag_kind='kde')
plt.show()

## 8. Kết luận & Insights từ EDA

**Quan sát chính:**
1. Glucose là biến quan trọng nhất (tương quan cao nhất với Outcome ~0.47)
2. BMI, Age, DiabetesPedigreeFunction cũng có ảnh hưởng rõ rệt
3. Nhiều giá trị 0 ở Glucose, BMI, Insulin → cần xử lý missing values (thay bằng median)
4. Có outlier ở Insulin, DiabetesPedigreeFunction, BMI → có thể cần xử lý (clip hoặc log transform)
5. Class imbalance ~65% - 35% → nên dùng class_weight='balanced' hoặc SMOTE
6. Không có missing value NaN thật, chỉ có giá trị 0 không hợp lý

**Hành động preprocessing đề xuất:**
- Thay 0 → NaN ở Glucose, BloodPressure, SkinThickness, Insulin, BMI
- Impute bằng median
- Scale dữ liệu (StandardScaler)
- Xử lý outlier nếu cần (ví dụ clip tại percentile 99)
- Giữ nguyên Pregnancies và Age (có 0 là hợp lý)